<p align="center">
  <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/4/4d/Logo-gustave-roussy.jpg/1200px-Logo-gustave-roussy.jpg" alt="Logo 1" width="250"/>
  <img src="https://upload.wikimedia.org/wikipedia/en/thumb/3/3f/Qube_Research_%26_Technologies_Logo.svg/1200px-Qube_Research_%26_Technologies_Logo.svg.png" alt="Logo 2" width="200" style="margin-left: 20px;"/>
</p>

# Data Challenge : Leukemia Risk Prediction


*GOAL OF THE CHALLENGE and WHY IT IS IMPORTANT:*

The goal of the challenge is to **predict disease risk for patients with blood cancer**, in the context of specific subtypes of adult myeloid leukemias.

The risk is measured through the **overall survival** of patients, i.e. the duration of survival from the diagnosis of the blood cancer to the time of death or last follow-up.

Estimating the prognosis of patients is critical for an optimal clinical management. 
For exemple, patients with low risk-disease will be offered supportive care to improve blood counts and quality of life, while patients with high-risk disease will be considered for hematopoietic stem cell transplantion.

The performance metric used in the challenge is the **IPCW-C-Index**.

*THE DATASETS*

The **training set is made of 3,323 patients**.

The **test set is made of 1,193 patients**.

For each patient, you have acces to CLINICAL data and MOLECULAR data.

The details of the data are as follows:

- OUTCOME:
  * OS_YEARS = Overall survival time in years
  * OS_STATUS = 1 (death) , 0 (alive at the last follow-up)

- CLINICAL DATA, with one line per patient:
  
  * ID = unique identifier per patient
  * CENTER = clinical center
  * BM_BLAST = Bone marrow blasts in % (blasts are abnormal blood cells)
  * WBC = White Blood Cell count in Giga/L 
  * ANC = Absolute Neutrophil count in Giga/L
  * MONOCYTES = Monocyte count in Giga/L
  * HB = Hemoglobin in g/dL
  * PLT = Platelets coutn in Giga/L
  * CYTOGENETICS = A description of the karyotype observed in the blood cells of the patients, measured by a cytogeneticist. Cytogenetics is the science of chromosomes. A karyotype is performed from the blood tumoral cells. The convention for notation is ISCN (https://en.wikipedia.org/wiki/International_System_for_Human_Cytogenomic_Nomenclature). Cytogenetic notation are: https://en.wikipedia.org/wiki/Cytogenetic_notation. Note that a karyotype can be normal or abnornal. The notation 46,XX denotes a normal karyotype in females (23 pairs of chromosomes including 2 chromosomes X) and 46,XY in males (23 pairs of chromosomes inclusing 1 chromosme X and 1 chromsome Y). A common abnormality in the blood cancerous cells might be for exemple a loss of chromosome 7 (monosomy 7, or -7), which is typically asssociated with higher risk disease

- GENE MOLECULAR DATA, with one line per patient per somatic mutation. Mutations are detected from the sequencing of the blood tumoral cells. 
We call somatic (= acquired) mutations the mutations that are found in the tumoral cells but not in other cells of the body.

  * ID = unique identifier per patient
  * CHR START END = position of the mutation on the human genome
  * REF ALT = reference and alternate (=mutant) nucleotide
  * GENE = the affected gene
  * PROTEIN_CHANGE = the consequence of the mutation on the protei that is expressed by a given gene
  * EFFECT = a broad categorization of the mutation consequences on a given gene.
  * VAF = Variant Allele Fraction = it represents the **proportion** of cells with the deleterious mutations. 

In [1]:
# Import necessary libraries
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from sklearn.tree import plot_tree
from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV
from sksurv.ensemble import RandomSurvivalForest
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.metrics import concordance_index_censored , concordance_index_ipcw
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sksurv.util import Surv

# Clinical Data
df = pd.read_csv(r"C:\Users\guill\Desktop\Data Challenge QRT\X_train\clinical_train.csv")
df_eval = pd.read_csv(r"C:\Users\guill\Desktop\Data Challenge QRT\X_test\clinical_test.csv")

# Molecular Data
maf_df = pd.read_csv(r"C:\Users\guill\Desktop\Data Challenge QRT\X_train\molecular_train.csv")
maf_eval = pd.read_csv(r"C:\Users\guill\Desktop\Data Challenge QRT\X_test\molecular_test.csv")

target_df = pd.read_csv(r"C:\Users\guill\Desktop\Data Challenge QRT\target_train.csv")
target_df_test = pd.read_csv(r"C:\Users\guill\Desktop\Data Challenge QRT\random_submission_FRacdcw_v9kP4pP.csv")

# Preview the data
df.head()

,ID,CENTER,BM_BLAST,WBC,ANC,MONOCYTES,HB,PLT,CYTOGENETICS
0,P132697,MSK,14.0,2.8,0.2,0.7,7.6,119.0,"46,xy,del(20)(q12)[2]/46,xy[18]"
1,P132698,MSK,1.0,7.4,2.4,0.1,11.6,42.0,"46,xx"
2,P116889,MSK,15.0,3.7,2.1,0.1,14.2,81.0,"46,xy,t(3;3)(q25;q27)[8]/46,xy[12]"
3,P132699,MSK,1.0,3.9,1.9,0.1,8.9,77.0,"46,xy,del(3)(q26q27)[15]/46,xy[5]"
4,P132700,MSK,6.0,128.0,9.7,0.9,11.1,195.0,"46,xx,t(3;9)(p13;q22)[10]/46,xx[10]"


### Step 1: Data Preparation (clinical data only)

For survival analysis, we’ll format the dataset so that OS_YEARS represents the time variable and OS_STATUS represents the event indicator.

In [2]:
# Drop rows where 'OS_YEARS' is NaN if conversion caused any issues
target_df.dropna(subset=['OS_YEARS', 'OS_STATUS'], inplace=True)

# Check the data types to ensure 'OS_STATUS' is boolean and 'OS_YEARS' is numeric
print(target_df[['OS_STATUS', 'OS_YEARS']].dtypes)

# Contarget_dfvert 'OS_YEARS' to numeric if it isn’t already
target_df['OS_YEARS'] = pd.to_numeric(target_df['OS_YEARS'], errors='coerce')

# Ensure 'OS_STATUS' is boolean
target_df['OS_STATUS'] = target_df['OS_STATUS'].astype(bool)

# Select features: BM_BLAST = % de blastes dans la moelle, HB = hémoglobine, PLT = plaquettes -> on construit un mini-modèle
features = ['BM_BLAST', 'HB', 'PLT']
target = ['OS_YEARS', 'OS_STATUS']

# Create the survival data format
X = df.loc[df['ID'].isin(target_df['ID']), features]
y = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', target_df)

OS_STATUS    float64
OS_YEARS     float64
dtype: object


### Step 2: Splitting the Dataset
We’ll split the data into training and testing sets to evaluate the model’s performance.

In [3]:
# Split the data into training and testing sets: on prend 70% des données pour entraîner, et 30% pour tester
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [4]:
# Survival-aware imputation for missing values: on remplace les éventuelles valeurs manquantes par la médiane
imputer = SimpleImputer(strategy="median")
X_train[['BM_BLAST', 'HB', 'PLT']] = imputer.fit_transform(X_train[['BM_BLAST', 'HB', 'PLT']])
X_test[['BM_BLAST', 'HB', 'PLT']] = imputer.transform(X_test[['BM_BLAST', 'HB', 'PLT']])

### Modèle Random Survival Forest

Random Survival Forest (RSF) est souvent le meilleur compromis pour débuter en survie :

✔ gère naturellement la censure

✔ gère les non-linéarités

✔ gère les interactions complexes

✔ pas d’hypothèse de proportionnalité des risques (contrairement à Cox)

✔ robuste aux données bruitées ou manquantes

✔ très adapté à une taille d’échantillon de 3000 patients

In [5]:
rsf = RandomSurvivalForest(
    n_estimators=600,          # Nombre d’arbres
    min_samples_split=10,      # Régularisation
    min_samples_leaf=18,
    max_features=0.5,       # Comme un RF classique
    n_jobs=-1,                 # Utilise tous les CPU
    random_state=42
)

# Entraînement
rsf.fit(X_train, y_train)

RandomSurvivalForest(max_features=0.5, min_samples_leaf=18,
                     min_samples_split=10, n_estimators=600, n_jobs=-1,
                     random_state=42)

In [6]:
# Prédictions
pred_train = rsf.predict(X_train)
pred_test = rsf.predict(X_test)

In [7]:
# Évaluation (C-index)
cindex_train = concordance_index_censored(
    y_train["OS_STATUS"], y_train["OS_YEARS"], pred_train
)[0]

cindex_test = concordance_index_censored(
    y_test["OS_STATUS"], y_test["OS_YEARS"], pred_test
)[0]

print(f"RSF C-index train : {cindex_train:.3f}")
print(f"RSF C-index test  : {cindex_test:.3f}")

RSF C-index train : 0.728
RSF C-index test  : 0.697


Au début j'étais parti sur n_estimators à 500, min_sample_leaf à 5, et max_features "sqrt" et ça donnait score train : 0.793, test : 0.691.
Donc le modèle overfit un peu.
Donc j'ai augmenté à 10 le min_sample_leaf à 10, ça a réduit l'overfitting et donné train : 0.752, test : 0.696.

### Step 5: Naive Approach to Incorporate Mutations

In this step, we take a very naive approach to account for genetic mutations by simply counting the total number of somatic mutations per patient. Instead of analyzing specific mutations or their biological impact, we use this aggregate count as a basic feature to reflect the mutational burden for each individual. Although simplistic, this feature can serve as a general indicator of genetic variability across patients, which may influence survival outcomes. More sophisticated mutation analysis could be incorporated in future models to improve predictive power.


In [8]:
# Step: Extract the number of somatic mutations per patient
# Group by 'ID' and count the number of mutations (rows) per patient
tmp = maf_df.groupby('ID').size().reset_index(name='Nmut')

# Merge with the training dataset and replace missing values in 'Nmut' with 0
df_2 = df.merge(tmp, on='ID', how='left').fillna({'Nmut': 0})

In [9]:
# Select features
features = ['BM_BLAST', 'HB', 'PLT', 'Nmut']
target = ['OS_YEARS', 'OS_STATUS']

# Create the survival data format
X = df_2.loc[df_2['ID'].isin(target_df['ID']), features]
y = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', target_df)

In [10]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [11]:
# Survival-aware imputation for missing values
imputer = SimpleImputer(strategy="median")
X_train[['BM_BLAST', 'HB', 'PLT', 'Nmut']] = imputer.fit_transform(X_train[['BM_BLAST', 'HB', 'PLT', 'Nmut']])
X_test[['BM_BLAST', 'HB', 'PLT', 'Nmut']] = imputer.transform(X_test[['BM_BLAST', 'HB', 'PLT', 'Nmut']])

### Ajout des mutations

In [12]:
# Maintenant on rajoute les mutations dans le RSF :

rsf = RandomSurvivalForest(
    n_estimators=600,
    min_samples_split=10,
    min_samples_leaf=18,
    max_features=0.5,
    n_jobs=-1,
    random_state=42
)

rsf.fit(X_train, y_train)

pred_train = rsf.predict(X_train)
pred_test = rsf.predict(X_test)


c_train = concordance_index_censored(y_train["OS_STATUS"], y_train["OS_YEARS"], pred_train)[0]
c_test = concordance_index_censored(y_test["OS_STATUS"], y_test["OS_YEARS"], pred_test)[0]

print(f"RSF C-index train: {c_train:.3f}")
print(f"RSF C-index test:  {c_test:.3f}")


RSF C-index train: 0.751
RSF C-index test:  0.715


### Construction de features moléculaires propres et robustes (top 20 gènes + VAF stats + Nmut)

In [13]:
# ==== 1. Préparation moléculaire ====

# Compte total de mutations par patient
mut_count = maf_df.groupby("ID").size().rename("Nmut")

# Stats VAF par patient
vaf_stats = maf_df.groupby("ID").agg(
    VAF_mean=("VAF", "mean"),
    VAF_max=("VAF", "max"),
    VAF_count=("VAF", "count")
)

# Remplir les NaN (patients sans mutation)
vaf_stats = vaf_stats.fillna(0)

In [14]:
# ==== 2. Identifier les 20 gènes les plus mutés ====

# Nombre de mutations par gène
top_genes = (
    maf_df["GENE"].value_counts()
    .head(20)
    .index
    .tolist()
)

print("Top 20 genes:", top_genes)

# Création d’un pivot ID x GENE (1 = mutation)
gene_matrix = (
    maf_df[maf_df["GENE"].isin(top_genes)]
    .assign(mut=1)
    .pivot_table(
        index="ID", columns="GENE", values="mut",
        aggfunc="max", fill_value=0
    )
)

# Renommer colonnes : GENE_mut
gene_matrix = gene_matrix.add_suffix("_mut")

Top 20 genes: ['TET2', 'ASXL1', 'SF3B1', 'DNMT3A', 'RUNX1', 'SRSF2', 'TP53', 'STAG2', 'U2AF1', 'EZH2', 'CBL', 'BCOR', 'NRAS', 'ZRSR2', 'DDX41', 'IDH2', 'CUX1', 'NF1', 'PHF6', 'KRAS']


In [15]:
# ==== 3. Fusionner les features moléculaires ====

mol_features = (
    pd.concat([mut_count, vaf_stats, gene_matrix], axis=1)
    .fillna(0)
)

# Fusion avec les données cliniques
df_full = df.merge(mol_features, on="ID", how="left")

# Remplacer NaN restants
df_full = df_full.fillna(0)

In [16]:
# ==== 4. Construction du jeu survival ====

# Liste des features finales
feature_cols = [
    "BM_BLAST", "HB", "PLT",
    "Nmut", "VAF_mean", "VAF_max", "VAF_count"
] + list(gene_matrix.columns)

X = df_full.loc[df_full['ID'].isin(target_df['ID']), feature_cols]
y = Surv.from_dataframe("OS_STATUS", "OS_YEARS", target_df)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Imputations (au cas où)
imputer = SimpleImputer(strategy="median")
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

In [17]:
rsf = RandomSurvivalForest(
    n_estimators=895,
    min_samples_split=8,
    min_samples_leaf=23,
    max_features=0.31,
    max_depth=13,
    random_state=42,
    n_jobs=-1
)

rsf.fit(X_train, y_train)

# Prédictions
pred_train = rsf.predict(X_train)
pred_test = rsf.predict(X_test)

# C-index
c_train = concordance_index_censored(
    y_train["OS_STATUS"], y_train["OS_YEARS"], pred_train
)[0]

c_test = concordance_index_censored(
    y_test["OS_STATUS"], y_test["OS_YEARS"], pred_test
)[0]

print(f"C-index train : {c_train:.3f}")
print(f"C-index test  : {c_test:.3f}")

C-index train : 0.766
C-index test  : 0.731


On obtient 0.732 avec les paramètres 500, 10, 5, sqrt
On obtient 0.731 avec 600, 10, 18, 0.5
On obtient 0.731 avec Optuna, 895, 8, 23, 0.31 et max_depth 13

In [18]:
"""
import optuna

def objective(trial):

    # --- Hyperparameters suggestions ---
    n_estimators = trial.suggest_int("n_estimators", 200, 1200)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 2, 50)
    min_samples_split = trial.suggest_int("min_samples_split", 4, 50)
    
    max_features = trial.suggest_float("max_features", 0.3, 1.0)

    # Optional:
    max_depth = trial.suggest_int("max_depth", 3, 15)

    # --- Model ---
    model = RandomSurvivalForest(
        n_estimators=n_estimators,
        min_samples_leaf=min_samples_leaf,
        min_samples_split=min_samples_split,
        max_features=max_features,
        max_depth=max_depth,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    # --- Predict risk scores ---
    pred = model.predict(X_test)

    # --- Compute C-index ---
    cindex = concordance_index_censored(
        y_test["OS_STATUS"], y_test["OS_YEARS"], pred
    )[0]

    return cindex

# --- Run Optuna study ---
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=40, show_progress_bar=True)

print("Best C-index:", study.best_value)
print("Best params:", study.best_params)
"""

'\nimport optuna\n\ndef objective(trial):\n\n    # --- Hyperparameters suggestions ---\n    n_estimators = trial.suggest_int("n_estimators", 200, 1200)\n    min_samples_leaf = trial.suggest_int("min_samples_leaf", 2, 50)\n    min_samples_split = trial.suggest_int("min_samples_split", 4, 50)\n    \n    max_features = trial.suggest_float("max_features", 0.3, 1.0)\n\n    # Optional:\n    max_depth = trial.suggest_int("max_depth", 3, 15)\n\n    # --- Model ---\n    model = RandomSurvivalForest(\n        n_estimators=n_estimators,\n        min_samples_leaf=min_samples_leaf,\n        min_samples_split=min_samples_split,\n        max_features=max_features,\n        max_depth=max_depth,\n        random_state=42,\n        n_jobs=-1\n    )\n\n    model.fit(X_train, y_train)\n\n    # --- Predict risk scores ---\n    pred = model.predict(X_test)\n\n    # --- Compute C-index ---\n    cindex = concordance_index_censored(\n        y_test["OS_STATUS"], y_test["OS_YEARS"], pred\n    )[0]\n\n    return 

### Modèle plus complexe : ajout de la cytogénétique + gènes utiles

In [21]:
# ---- 1. Mutation burden (déjà fait) ----
tmp = maf_df.groupby("ID").size().reset_index(name="Nmut")
df2 = df.merge(tmp, on="ID", how="left").fillna({"Nmut": 0})


# ---- 2. CYTOGENETICS ENCODING ----

def cytogenetic_flags(s):
    s = str(s).lower()
    return pd.Series({
        "cyto_normal": int("normal" in s),
        "cyto_complex": int("complex" in s or "+mar" in s),
        "cyto_del5q": int("del(5" in s or "5q" in s),
        "cyto_monosomy5": int("-5" in s),
        "cyto_monosomy7": int("-7" in s),
        "cyto_trisomy8": int("+8" in s),
        "cyto_11q23": int("11q23" in s or "kmt2a" in s),
    })

cyto_features = df["CYTOGENETICS"].apply(cytogenetic_flags)
df3 = pd.concat([df2, cyto_features], axis=1)


# ---- 3. GENE ENCODING ----

# Liste des gènes importants
key_genes = [
    "TP53","NPM1","FLT3","DNMT3A","IDH1","IDH2","TET2","RUNX1",
    "ASXL1","NRAS","KRAS","CBL",
    "SF3B1","SRSF2","U2AF1","ZRSR2"
]

# Create a pivot table: 1 = mutated, 0 = not mutated
gene_table = (
    maf_df.assign(mut=1)
    .pivot_table(index="ID", columns="GENE", values="mut", aggfunc="max")
    .fillna(0)
)

# Keep only useful genes
gene_table = gene_table.reindex(columns=key_genes, fill_value=0)

# Merge into clinical data
df_final = df3.merge(gene_table, on="ID", how="left").fillna(0)


# =======================================
# Final feature set
# =======================================

features = (
    ['BM_BLAST','HB','PLT','Nmut'] +
    list(cyto_features.columns) +
    key_genes
)

X = df_final.loc[df_final['ID'].isin(target_df['ID']), features]
y = Surv.from_dataframe("OS_STATUS", "OS_YEARS", target_df)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Imputation
imputer = SimpleImputer(strategy="median")
X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
X_test = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)



In [22]:
rsf = RandomSurvivalForest(
    n_estimators=895,
    min_samples_split=8,
    min_samples_leaf=23,
    max_features=0.31,
    max_depth=13,
    random_state=42,
    n_jobs=-1
)

rsf.fit(X_train, y_train)

# Prédictions
pred_train = rsf.predict(X_train)
pred_test = rsf.predict(X_test)

# C-index
c_train = concordance_index_censored(
    y_train["OS_STATUS"], y_train["OS_YEARS"], pred_train
)[0]

c_test = concordance_index_censored(
    y_test["OS_STATUS"], y_test["OS_YEARS"], pred_test
)[0]

print(f"C-index train : {c_train:.3f}")
print(f"C-index test  : {c_test:.3f}")

C-index train : 0.756
C-index test  : 0.736


C'ets déjà pas mal, on passe à 0.736.
Mais peut-être que RSF marche moins bien dans ce contexte que LGBM ou Cox, il faudra voir

On va mieux gérer les critères de cytogénétique en les parsant selon ELN 2022 (fav, adverse, intermediate)

In [35]:
import re

def classify_eln(cytogen):
    if pd.isna(cytogen):
        return "Unknown"
    
    c = cytogen.upper().replace(" ", "")
    
    # ---- Favorable ----
    if re.search(r"t\(8;21\)", c):
        return "Favorable"
    if re.search(r"inv\(16\)|t\(16;16\)", c):
        return "Favorable"
    if re.search(r"t\(15;17\)", c):
        return "Favorable"

    # ---- Adverse ----
    if re.search(r"DEL\(5\)|-5|5Q", c):
        return "Adverse"
    if re.search(r"-7|\bDEL\(7\)", c):
        return "Adverse"
    if re.search(r"17P|DEL\(17\)", c):
        return "Adverse"
    if re.search(r"INV\(3\)|T\(3;3\)", c):
        return "Adverse"
    if re.search(r"T\(6;9\)", c):
        return "Adverse"
    
    # complexe karyotype
    # Si ≥3 anomalies listées par /
    if "/" in cytogen and len(cytogen.split("/")) >= 3:
        return "Adverse"

    # ---- Intermediate ----
    if "NORMAL" in c:
        return "Intermediate"
    if re.search(r"\+8", c):
        return "Intermediate"

    # ---- Default ----
    return "Intermediate"

df4 = df3.copy()

df4["ELN_RISK"] = df4["CYTOGENETICS"].apply(classify_eln)

df4 = pd.get_dummies(df4, columns=["ELN_RISK"], drop_first=True)



In [33]:
features = (
    ['BM_BLAST', 'HB', 'PLT', 'Nmut'] +
    key_genes +
    ['ELN_RISK', 'ELN_RISK']
)


X = df4.loc[df4['ID'].isin(target_df['ID']), features]
y = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', target_df)


# ============================================================
# 6. Split train/test
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Imputation médiane
imputer = SimpleImputer(strategy="median")
X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
X_test = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

KeyError: "['TP53', 'NPM1', 'FLT3', 'DNMT3A', 'IDH1', 'IDH2', 'TET2', 'RUNX1', 'ASXL1', 'NRAS', 'KRAS', 'CBL', 'SF3B1', 'SRSF2', 'U2AF1', 'ZRSR2', 'ELN_RISK'] not in index"

In [ ]:
rsf = RandomSurvivalForest(
    n_estimators=800,
    min_samples_split=10,
    min_samples_leaf=8,
    max_features="sqrt",
    n_jobs=-1,
    random_state=42
)

rsf.fit(X_train, y_train)

pred_train = rsf.predict(X_train)
pred_test = rsf.predict(X_test)

cindex_train = concordance_index_censored(
    y_train["event"], y_train["time"], pred_train
)[0]

cindex_test = concordance_index_censored(
    y_test["event"], y_test["time"], pred_test
)[0]

print(f"RSF C-index train : {cindex_train:.3f}")
print(f"RSF C-index test  : {cindex_test:.3f}")

### Inference on test set

In [19]:

tmp_eval = maf_eval.groupby('ID').size().reset_index(name='Nmut')

# Merge with the training dataset and replace missing values in 'Nmut' with 0
df_eval = df_eval.merge(tmp_eval, on='ID', how='left').fillna({'Nmut': 0})



In [20]:

df_eval[['BM_BLAST', 'HB', 'PLT', 'Nmut']] = imputer.transform(df_eval[['BM_BLAST', 'HB', 'PLT', 'Nmut']])

prediction_on_test_set = cox.predict(df_eval.loc[:, features])

ValueError: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- ASXL1_mut
- BCOR_mut
- CBL_mut
- CUX1_mut
- DDX41_mut
- ...


In [ ]:
prediction_on_test_set

array([ 0.86383926, -0.51383953, -1.73261271, ..., -1.69091907,
       -1.44228073, -1.5665999 ])

In [ ]:
submission = pd.Series(prediction_on_test_set, index=df_eval['ID'], name='OS_YEARS')

In [ ]:
submission

ID
KYW1       0.863839
KYW2      -0.513840
KYW3      -1.732613
KYW4       0.493820
KYW5      -1.125188
             ...   
KYW1189   -1.566600
KYW1190   -1.442281
KYW1191   -1.690919
KYW1192   -1.442281
KYW1193   -1.566600
Name: OS_YEARS, Length: 1193, dtype: float64

In [ ]:

submission.to_csv('./benchmark_submission.csv')

In [ ]:
submission

ID
KYW1       0.863839
KYW2      -0.513840
KYW3      -1.732613
KYW4       0.493820
KYW5      -1.125188
             ...   
KYW1189   -1.566600
KYW1190   -1.442281
KYW1191   -1.690919
KYW1192   -1.442281
KYW1193   -1.566600
Name: OS_YEARS, Length: 1193, dtype: float64

In [ ]:

random_submission = pd.Series(np.random.uniform(0, 1, len(submission)),index =submission.index, name='OS_YEARS')


In [ ]:
random_submission.to_csv('./random_submission.csv')

In [ ]:
random_submission

ID
KYW1       0.293724
KYW2       0.756793
KYW3       0.823897
KYW4       0.241738
KYW5       0.969231
             ...   
KYW1189    0.401949
KYW1190    0.267835
KYW1191    0.514027
KYW1192    0.909906
KYW1193    0.103759
Name: OS_YEARS, Length: 1193, dtype: float64